In [1]:
from neuron import h
import matplotlib.pyplot as plt
import numpy as np
import plotly
import matplotlib
import pandas as pd

# Determine the minimum required electric field strength to elicit action potentials

TODO: run with AMPA receptors and without NMDA receptors? (without the synaptic plasticity mechanism)
- But there is current flowing through NMDA receptors too?
- 

In [2]:
h.load_file("main.hoc")
# h.load_file("ampa_nmda_ltp.hoc")

	1 
	1 
	1 
	1 
	0 
	1 
23 
	1 
	1 
After any change to cell geometry or nseg, be sure to invoke setpointers()
	1 
	1 
	1 
	0 
	0 
	0 
	0 
	0 
	0 
	1 
Use setstim(DEL, DUR, F0, F1, AMP, PHASE) to change latency (ms), duration (ms),
start and end frequency (Hz), amplitude (V/m) and phase (degrees) of applied
electrical field.
	0 
	0 
	0 
	0 
	0 
	0 
	0 
	0 
	1 
	1 
	1 
	0 
	0 
	0 
	0 
	0 
	0 


1.0

In [3]:
def set_stimulus(delay = 100, duration = 900, frequency1 = 1000, frequency2 = 1005, amplitude = 100, phase = 0):
    h.setstim(delay, duration, frequency1, frequency2, amplitude, phase)

In [4]:
def run(tstop = 100, delay = 100, duration = 900, frequency1 = 1000, frequency2 = 1005, amplitude = 100, phase = 0):
    # Set stimulation time
    h.tstop = tstop

    # Setup for recording membrane potential in the soma and time
    soma_v = h.Vector().record(h.soma[0](0.5)._ref_v)
    t = h.Vector().record(h._ref_t)

    set_stimulus(
        delay = delay,
        duration = duration,
        frequency1 = frequency1,
        frequency2 = frequency2,
        amplitude = amplitude,
        phase = phase
    )

    h.run()

    return pd.DataFrame({
        "Time": t,
        "Voltage": soma_v
    })

In [5]:
def has_ap(carrier, target_frequency, amplitude, ap_threshold = -30):
    stim_res = run(
        tstop = 500,
        delay = 0,
        duration = 500,
        frequency1 = carrier,
        frequency2 = carrier + target_frequency,
        amplitude = amplitude,
        phase = 0
    )

    return np.any(stim_res.Voltage >= ap_threshold)

In [6]:
def find_ef_amplitude(carrier, target = 0, initial_amplitude = 1000.0, ap_threshold = -30, epsilon = 1):
    amplitude = initial_amplitude
    amplitude_step = amplitude / 2
    last_amplitude = initial_amplitude * 2

    while abs(last_amplitude - amplitude) > epsilon:
        print(f"Testing {amplitude}")
        last_amplitude = amplitude

        if has_ap(carrier, target, amplitude, ap_threshold = ap_threshold):
            amplitude -= amplitude_step
        else:
            amplitude += amplitude_step

        amplitude_step = amplitude_step / 2

    print(f"Minimum amplitude for carrier {carrier} with target {target}: {amplitude}")
    return amplitude

In [11]:
carriers = [5, 100, 1000, 1100]

res = []
for carrier in carriers:
    print(f"Starting carrier {carrier}")
    amplitude = find_ef_amplitude(carrier)

    res.append(pd.DataFrame({
        "Carrier": [carrier],
        "Amplitude": [amplitude]
    }))

res = pd.concat(res)
res.reset_index(drop = True, inplace = True)

Starting carrier 5
Testing 1000.0
Testing 500.0
Testing 250.0
Testing 125.0
Testing 62.5
Testing 31.25
Testing 15.625
Testing 23.4375
Testing 19.53125
Testing 21.484375
Minimum amplitude for carrier 5 with target 0: 22.4609375
Starting carrier 100
Testing 1000.0
Testing 500.0
Testing 250.0
Testing 125.0
Testing 62.5
Testing 31.25
Testing 46.875
Testing 54.6875
Testing 50.78125
Testing 48.828125
Minimum amplitude for carrier 100 with target 0: 47.8515625
Starting carrier 1000
Testing 1000.0
Testing 500.0
Testing 250.0
Testing 125.0
Testing 187.5
Testing 156.25
Testing 140.625
Testing 148.4375
Testing 152.34375
Testing 150.390625
Minimum amplitude for carrier 1000 with target 0: 149.4140625
Starting carrier 1100
Testing 1000.0
Testing 500.0
Testing 250.0
Testing 125.0
Testing 187.5
Testing 156.25
Testing 171.875
Testing 164.0625
Testing 160.15625
Testing 158.203125
Minimum amplitude for carrier 1100 with target 0: 157.2265625


In [12]:
res

,Carrier,Amplitude
0,5,22.460938
1,100,47.851562
2,1000,149.414062
3,1100,157.226562
